# 06. Modeling
**목적**: label 유무에 따라 파이프라인을 분기한다.

- **Case A (label 있음)**: LightGBM / XGBoost supervised 분류 모델
- **Case B (label 없음)**: Risk Index 기반 우선순위화 + HDBSCAN 클러스터링

두 경우 모두 07_explainability 에서 사용할 결과물을 저장한다.

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from config import DATA_PROCESSED, OUT_TABLES, SEED

with open(DATA_PROCESSED / 'column_map.json', encoding='utf-8') as f:
    cmap = json.load(f)
FLAGS  = cmap['flags']
FC     = cmap['facility']
FID_COL = FC['facility_id']

df = pd.read_parquet(DATA_PROCESSED / 'risk_scores.parquet')
df_buf = pd.read_parquet(DATA_PROCESSED / 'buffer_features.parquet')
df_weather = pd.read_parquet(DATA_PROCESSED / 'weather_features.parquet')

print(f'모델 유형: {"Case A" if FLAGS["has_label"] else "Case B"}')
print(f'데이터 크기: {df.shape}')

## Feature Matrix 구성

In [ ]:
# 기상 feature 컬럼
weather_cols = [c for c in df_weather.columns
                if c not in [FID_COL, 'date', 'stn_id', 'nearest_stn_id']
                and df_weather[c].dtype in [np.float64, np.int64]]

# buffer feature 컬럼
buf_cols = [c for c in df_buf.columns if c != FID_COL]

# 전체 feature matrix 구성
df_feat = df[[FID_COL, 'date', 'final_risk', 'risk_grade',
              'weather_hazard', 'spatial_exposure', 'facility_exposure', 'hist_prior']].copy()

df_feat = df_feat.merge(df_weather[[FID_COL, 'date'] + weather_cols], on=[FID_COL, 'date'], how='left')
df_feat = df_feat.merge(df_buf[[FID_COL] + buf_cols], on=FID_COL, how='left')

print(f'Feature matrix: {df_feat.shape}')

In [ ]:
FEATURE_COLS = [c for c in df_feat.columns
                if c not in [FID_COL, 'date', 'final_risk', 'risk_grade',
                             'risk_grade_quantile']
                and df_feat[c].dtype in [np.float64, np.float32, np.int64, np.int32]]

X = df_feat[FEATURE_COLS].fillna(df_feat[FEATURE_COLS].median())
print(f'Feature 수: {len(FEATURE_COLS)}')
print(FEATURE_COLS[:10], '...')

---
## Case A — Supervised ML (label 있을 때)

In [ ]:
if FLAGS['has_label']:
    from sklearn.model_selection import StratifiedKFold
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score
    import lightgbm as lgb
    import xgboost as xgb

    LABEL_COL = FC['label']
    y = df_feat[LABEL_COL].fillna(0).astype(int)

    print(f'Label 분포: \n{y.value_counts()}')

In [ ]:
if FLAGS['has_label']:
    # Time-based split (가장 최근 20%를 test로)
    dates_sorted = df_feat['date'].sort_values().unique()
    cutoff = dates_sorted[int(len(dates_sorted) * 0.8)]

    train_mask = df_feat['date'] < cutoff
    test_mask  = df_feat['date'] >= cutoff

    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
    print(f'Test 기간: {df_feat.loc[test_mask, "date"].min()} ~ {df_feat.loc[test_mask, "date"].max()}')

In [ ]:
if FLAGS['has_label']:
    results = {}

    models = {
        'logistic': LogisticRegression(max_iter=1000, random_state=SEED),
        'random_forest': RandomForestClassifier(n_estimators=200, random_state=SEED),
        'lightgbm': lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                        num_leaves=63, random_state=SEED, verbose=-1),
        'xgboost': xgb.XGBClassifier(n_estimators=500, learning_rate=0.05,
                                      random_state=SEED, eval_metric='logloss',
                                      verbosity=0),
    }

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)

        # Recall@Top-K
        k10 = max(1, int(len(y_test) * 0.10))
        top_k_idx = np.argsort(y_prob)[::-1][:k10]
        recall_topk = y_test.values[top_k_idx].sum() / max(1, y_test.sum())

        results[name] = {
            'AUC': roc_auc_score(y_test, y_prob),
            'F1':  f1_score(y_test, y_pred, zero_division=0),
            'Recall': recall_score(y_test, y_pred, zero_division=0),
            'Precision': precision_score(y_test, y_pred, zero_division=0),
            'Recall@Top10%': recall_topk,
        }
        print(f'{name}: AUC={results[name]["AUC"]:.3f}  Recall@Top10%={recall_topk:.3f}')

    df_results = pd.DataFrame(results).T
    print('\n=== 모델 성능 비교 ===')
    print(df_results.round(3))
    df_results.to_csv(OUT_TABLES / 'model_performance.csv')

    # 최종 모델 선택 (AUC 기준)
    best_name = df_results['AUC'].idxmax()
    best_model = models[best_name]
    print(f'\n최종 모델: {best_name}')

    import pickle
    with open(DATA_PROCESSED / 'best_model.pkl', 'wb') as f:
        pickle.dump({'model': best_model, 'feature_cols': FEATURE_COLS, 'name': best_name}, f)
    print('모델 저장 완료: best_model.pkl')

---
## Case B — Risk Prioritisation + Clustering (label 없을 때)

In [ ]:
if not FLAGS['has_label']:
    print('Case B: Rule-based Risk Index + Spatial Clustering')

    # 최신 날짜 기준 설비별 위험도 요약
    latest = df.groupby(FID_COL).agg(
        final_risk_mean=('final_risk', 'mean'),
        final_risk_max=('final_risk', 'max'),
        final_risk_last=('final_risk', 'last'),
        high_risk_days=('risk_grade', lambda x: (x.isin(['High', 'Very High'])).sum()),
    ).reset_index()

    # 종합 우선순위 점수 (최근값 60% + 최대값 25% + 고위험일수 15%)
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler(feature_range=(0, 100))
    latest['priority_score'] = (
        0.60 * latest['final_risk_last'] +
        0.25 * latest['final_risk_max'] +
        0.15 * scaler.fit_transform(latest[['high_risk_days']]).flatten()
    ).clip(0, 100)

    latest = latest.sort_values('priority_score', ascending=False).reset_index(drop=True)
    latest['inspection_rank'] = latest.index + 1

    print(f'설비 수: {len(latest)}')
    print(latest.head(10))

In [ ]:
if not FLAGS['has_label']:
    import geopandas as gpd
    import hdbscan
    from config import CRS_PROJ

    gdf_fac = gpd.read_file(DATA_PROCESSED / 'facility_proj.gpkg')
    gdf_risk = gdf_fac.merge(latest, on=FID_COL, how='left')

    # 고위험 설비만 클러스터링 (Very High + High)
    gdf_high = gdf_risk[gdf_risk['final_risk_max'] >= 66].copy()

    if len(gdf_high) >= 10:
        coords = np.column_stack([gdf_high.geometry.x, gdf_high.geometry.y])
        clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
        gdf_high['cluster_id'] = clusterer.fit_predict(coords)

        print(f'고위험 설비 클러스터: {gdf_high["cluster_id"].nunique()}개')
        print(gdf_high['cluster_id'].value_counts())
    else:
        print(f'고위험 설비 수 부족 ({len(gdf_high)}개) — 클러스터링 생략')

    # 결과 저장
    latest.to_csv(OUT_TABLES / 'inspection_priority.csv', index=False)
    print('inspection_priority.csv 저장 완료')

    import pickle
    with open(DATA_PROCESSED / 'case_b_result.pkl', 'wb') as f:
        pickle.dump({'priority_df': latest, 'feature_cols': FEATURE_COLS}, f)
    print('case_b_result.pkl 저장 완료')

print('\n다음 단계: 07_explainability.ipynb')